# Post Call Metrics

基于 `ai_events`、`user_msgs` 与 `originality.csv`，围绕每次 AI 调用（click）构建**调用后行为特征**指标，并输出为 `post_call_metrics.csv`。  

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

import sys
sys.path.insert(0, '../../modules/semantic_toolbox')

from semdistance.distance import dis_sentences

In [ ]:
# 输入文件路径
pickle_dir = Path('../../data/pickle')
timeline_file = pickle_dir / 'timeline.pkl'
ai_events_file = pickle_dir / 'ai_events.pkl'
user_msgs_file = pickle_dir / 'user_msgs.pkl'

scoring_dir = Path('../../data/analysis/scoring')
originality_file = scoring_dir / 'originality.csv'

# 输出文件路径
output_dir = Path('../../data/analysis/post_call')
post_call_metrics_file = output_dir / 'post_call_metrics.csv'
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# 加载数据
timeline = pd.read_pickle(timeline_file)
ai_events = pd.read_pickle(ai_events_file)
user_msgs = pd.read_pickle(user_msgs_file)

In [ ]:
def compute_adoption_metrics(assistant_content, after_contents, prev_last_content, dis_func, sim_threshold=0.8, ratio_threshold=0.8):
    """
    计算采纳率相关指标
    
    参数：
        assistant_content: AI的回复内容
        after_contents: 调用后的用户回答列表（按时间顺序）
        prev_last_content: 调用前的最后一个用户回答（可为None）
        dis_func: 距离计算函数，调用方式 dis_func(s1, s2, 1)
        sim_threshold: 直接采纳的相似度阈值
        ratio_threshold: 间接采纳的比率阈值（mean_ai_distance / mean_prev_distance < ratio_threshold）
    
    返回：
        包含各项指标的字典
    """
    if not after_contents:
        return {
            'perspective_taking': np.nan,
            'affected_by_ai': np.nan,
            'mean_ai_distance': np.nan,
            'mean_prev_distance': np.nan,
            'adoption_ratio': np.nan
        }
    
    ai_distances = []
    prev_distances = []
    
    for content in after_contents:
        try:
            d_ai = dis_func(assistant_content, content, 1)
            ai_distances.append(d_ai)
            if prev_last_content:
                d_prev = dis_func(prev_last_content, content, 1)
                prev_distances.append(d_prev)
        except Exception as e:
            print(f'采纳率计算出错：{assistant_content} vs {content}，错误：{e}')
            continue
    
    # 初始化结果
    perspective_taking = np.nan
    mean_ai_distance = np.nan
    mean_prev_distance = np.nan
    adoption_ratio = np.nan
    affected_by_ai = np.nan
    
    if ai_distances:
        mean_ai_distance = np.mean(ai_distances)
        first_ai_dist = ai_distances[0]
        first_sim = 1 - first_ai_dist
        perspective_taking = 1 if first_sim > sim_threshold else 0
    
    if prev_distances:
        mean_prev_distance = np.mean(prev_distances)
        if mean_prev_distance > 0:
            # 后三个回答到 AI 回答的平均距离，比上它们到调用前第一个用户回答的平均距离，比值小于0.8
            adoption_ratio = mean_ai_distance / mean_prev_distance
            affected_by_ai = 1 if adoption_ratio < ratio_threshold else 0
    
    return {
        'perspective_taking': perspective_taking,
        'affected_by_ai': affected_by_ai,
        'mean_ai_distance': mean_ai_distance,
        'mean_prev_distance': mean_prev_distance,
        'adoption_ratio': adoption_ratio
    }

In [ ]:
# 原创性（新颖性）数据
originality_df = pd.read_csv(originality_file)
originality_df = originality_df[originality_df['trial_index'] != 0]  # 过滤掉练习试次
originality_series = originality_df.set_index(['participant_id', 'item_name', 'trial_index', 'time'])['originality']

post_call_metrics = []

for idx, row in ai_events.iterrows():
    pid = row['participant_id']
    item = row['item_name']
    trial = row['trial_index']
    click_time = row['click_time']
    assistant_content = row['assistant_content']
    
    # 调用前3个用户消息（时间上最近）
    prev_3 = user_msgs[
        (user_msgs['participant_id'] == pid) &
        (user_msgs['item_name'] == item) &
        (user_msgs['trial_index'] == trial) &
        (user_msgs['time'] < click_time)
    ].sort_values('time').tail(3)
    
    # 调用后3个用户消息（时间上之后）
    after_3 = user_msgs[
        (user_msgs['participant_id'] == pid) &
        (user_msgs['item_name'] == item) &
        (user_msgs['trial_index'] == trial) &
        (user_msgs['time'] > click_time)
    ].sort_values('time').head(3)
    
    # 获取前3条消息的原创性
    prev_originalities = []
    for _, msg in prev_3.iterrows():
        key = (pid, item, trial, msg['time'])
        if key in originality_series.index:
            prev_originalities.append(originality_series.loc[key])
        else:
            prev_originalities.append(np.nan)
    
    # 获取后3条消息的原创性
    after_originalities = []
    for _, msg in after_3.iterrows():
        key = (pid, item, trial, msg['time'])
        if key in originality_series.index:
            after_originalities.append(originality_series.loc[key])
        else:
            after_originalities.append(np.nan)

    # 1. originality提升量 = mean(after) - mean(prev)
    originality_boost = np.mean(after_originalities) - np.mean(prev_originalities) if len(prev_originalities) > 0 and len(after_originalities) > 0 else np.nan
    
    # 2. 语义距离跳跃
    last_before = prev_3.iloc[-1]['content'] if len(prev_3) > 0 else None
    first_after = after_3.iloc[0]['content'] if len(after_3) > 0 else None

    if last_before and first_after:
        try:
            semantic_jump = dis_sentences(last_before, first_after, 1)
            # 还需要计算后一个回复与AI回复的距离，来判断是否是AI回复引发的跳跃
            semantic_distance_to_ai = dis_sentences(assistant_content, first_after, 1)
        except Exception as e:
            print(f'ERROR: {last_before} vs {first_after}')
            semantic_jump = np.nan
            semantic_distance_to_ai = np.nan
    else:
        semantic_jump = np.nan
        semantic_distance_to_ai = np.nan
    
    # 3. 采纳/复用率
    # 提取调用后回答列表和调用前最后一个回答
    after_contents = after_3['content'].tolist() if not after_3.empty else []
    prev_last_content = last_before # 定义相同，复用前面计算过的

    # 4. 调用后思考时间
    # 此次AI点击后的第一个用户消息的时间 - AI点击时间
    if not after_3.empty:
        first_after_time = after_3.iloc[0]['time']
        after_ai_time = first_after_time - click_time
    else:
        after_ai_time = np.nan

    # 计算采纳率指标
    adopt_metrics = compute_adoption_metrics(
        assistant_content, 
        after_contents, 
        prev_last_content, 
        dis_sentences,
        sim_threshold=0.8,
        ratio_threshold=0.8
    )
    
    post_call_metrics.append({
        'participant_id': pid,
        'item_name': item,
        'trial_index': trial,
        'click_index': row['click_index'],
        'click_time': click_time,
        'originality_boost': originality_boost,
        'semantic_jump': semantic_jump,
        'semantic_distance_to_ai': semantic_distance_to_ai,
        'after_ai_time': after_ai_time,
        **adopt_metrics
    })

post_call_df = pd.DataFrame(post_call_metrics)
post_call_df.to_csv(post_call_metrics_file, index=False)